# Validation kualitas label: encoder kandidat + model label tetap

Hasil iPhone lengkap: MusicFM memperlihatkan percepatan warm sekitar 1,44× / 2,36× untuk profile 750 / 3000; MuQ sekitar 1,47× / 2,44×. Itu hasil encoder terpisah pada satu input per profile, belum percepatan seluruh aplikasi atau bukti kestabilan semua lagu.

Sekarang kita periksa **apakah kualitas label tetap terjaga** pada seluruh validation manual. Tidak ada training, perubahan boundary, perubahan label, pergantian head, atau perubahan checkpoint konteks epoch 5.

Kandidat statis hanya menerima window penuh 30/120 detik. Window akhir yang lebih pendek memakai embedding CPU palette6 lama dari cache yang telah divalidasi, **persis tanpa perubahan**. Tidak ada padding atau pemendekan konteks. Laporan mencatat berapa window/frame berasal dari masing-masing jalur. Ini menguji kombinasi kandidat statis + jalur partial lama, bukan penggantian semua panjang input dengan graph statis.

Encoder window penuh dan head palette6 berjalan pada **CPU Mac** untuk pemeriksaan kualitas; konteks tetap PyTorch yang sudah dibekukan. Belum ada perubahan di aplikasi. Hasil dibandingkan dengan pipeline palette6 saat ini serta konteks pada fitur training asli; delta kedua pembanding ditampilkan terpisah.


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib, json, subprocess

ROOT=Path.cwd().resolve()
assert (ROOT/'scripts/validate_encoder_pipeline.py').is_file(), 'Buka notebook dari root SongMap'
SONGSTRUCT_ROOT=ROOT.parent/'Section-Annotations'
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'  # Jangan resolve symlink executable venv.
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'

# Hash ekspor yang cocok dengan delapan benchmark iPhone yang sudah dikirim.
EXPECTED_EXPORT_SHA={
    'musicfm':'9f806441ac53e3f1732ac2fa578a5d28276e85e9af24eb507af96ed5cb6dfef0',
    'muq':'7505f6d40ecd93095d202d3d0f046af96693d245ab1d9414b557836438f3cb98',
}
EXPORTS={}
for role,expected in EXPECTED_EXPORT_SHA.items():
    paths=list((ROOT/'outputs.local').glob('encoder-acceleration-*/'+role+'/export-report.json'))
    EXPORTS[role]=next((p.resolve() for p in paths if hashlib.sha256(p.read_bytes()).hexdigest()==expected),None)
    assert EXPORTS[role] is not None, f'Ekspor {role} yang cocok dengan benchmark belum ditemukan; periksa folder outputs.local'

candidates=[]
for path in (ROOT/'outputs.local').glob('context-palette6-validation-*/validation-all.json'):
    r=json.loads(path.read_text())
    if r.get('full_validation') and r.get('selected_epoch')==5 and r.get('original_context_validation_reproduced'):
        candidates.append(path)
assert candidates, 'Validation palette6 konteks epoch 5 sebelumnya belum ditemukan'
PALETTE_VALIDATION=max(candidates,key=lambda p:p.stat().st_mtime).resolve()
selected=json.loads((PALETTE_VALIDATION.parent/'selection.json').read_text())
FROZEN=next((p.parent.resolve() for p in (ROOT/'outputs.local').glob('frozen-context-test-*/selection.json')
    if json.loads(p.read_text()).get('identity')==selected['frozen_selection_identity']),None)
assert FROZEN is not None, 'Snapshot konteks asli belum ditemukan'
BASELINE_POOL=ROOT/'outputs.local/section-context-palette6-cache-v1'
RESUME_OUTPUT=None  # Isi path encoder-pipeline-quality-* lama setelah restart kernel jika ingin melanjutkan.
OUTPUT=(Path(RESUME_OUTPUT).expanduser().resolve() if RESUME_OUTPUT else
    ROOT/'outputs.local'/('encoder-pipeline-quality-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f')))
subprocess.run([str(RUN_PYTHON),'-c','import sys,numpy,torch,coremltools,librosa; print("Python:",sys.executable); print("Torch:",torch.__version__); print("CoreMLTools:",coremltools.__version__)'],check=True)
print('Graph kandidat:',EXPORTS)
print('Pembanding:',PALETTE_VALIDATION)
print('Output:',OUTPUT)

def validate_quality(limit):
    command=[str(RUN_PYTHON),'-u',str(ROOT/'scripts/validate_encoder_pipeline.py'),
        '--songstruct-root',str(SONGSTRUCT_ROOT),'--songmap-root',str(ROOT),
        '--palette-validation',str(PALETTE_VALIDATION),'--frozen-evaluation',str(FROZEN),
        '--baseline-pool-cache',str(BASELINE_POOL),'--musicfm-export',str(EXPORTS['musicfm']),
        '--muq-export',str(EXPORTS['muq']),'--output-dir',str(OUTPUT),'--limit',str(limit)]
    log=Path(str(OUTPUT)+('.one.log' if limit else '.all.log'));log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a',encoding='utf-8') as handle:
        process=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:raise RuntimeError(f'Validation gagal ({status}); lihat {log}. Hasil track yang sudah lengkap dapat dipakai ulang.')


In [ ]:
# Satu lagu dahulu. Ini smoke check, belum ukuran kualitas seluruh validation.
validate_quality(1)


In [ ]:
# Seluruh validation, termasuk semua target manual layak pada tiap lagu.
# Hasil track pertama dipakai ulang jika identitasnya cocok; tidak menjalankan training.
validate_quality(0)


In [ ]:
from IPython.display import display, Markdown
report=json.loads((OUTPUT/'validation-all.json').read_text())
assert report['full_validation'] and not report['test_used']
lines=['| Pembanding/kandidat | Akurasi | Macro F1 |','|---|---|---|']
for name,key in [('Konteks pada fitur training asli','original_training_features_context'),
                 ('Pipeline palette6 saat ini','baseline_palette6_context'),('Encoder kandidat + partial lama','candidate')]:
    r=report[key]
    lines.append(f"| {name} | {r['section_accuracy']:.2%} | {r['macro_f1_present']:.6f} |")
display(Markdown('\n'.join(lines)))
for key in ['comparison_to_current_pipeline','comparison_to_original_context']:
    print('\n',key,':',json.dumps(report[key],indent=2))
print('\nKesamaan label dengan pipeline sekarang:',f"{report['section_agreement']:.2%}")
print('Jumlah section yang berubah:',len(report['changed_sections']))
print('Jalur window:',report['routes'])
print('Per-class F1 kandidat:',{k:round(v['f1'],4) for k,v in report['candidate']['per_class'].items()})
print('\nKirim file:',OUTPUT/'validation-all.json')


Validation ini menjalankan encoder window penuh lagi sehingga memerlukan waktu di Mac. Encoder untuk partial window tidak dijalankan ulang. Hasil ringkas tiap lagu disimpan (mean fitur section, prior label, prediksi dan provenance), tanpa membuat salinan cache frame seluruh dataset yang berukuran besar. Jika proses dihentikan, jalankan kembali cell yang sama; setelah restart kernel, isi `RESUME_OUTPUT` dengan folder output lama. Identitas sumber yang berubah membuat cache ditolak.

`no_observed_drop` hanya menyatakan bahwa akurasi, macro F1 dan F1 tiap kelas pada validation ini tidak turun terhadap pembanding tersebut. Nilai itu bukan jaminan untuk semua lagu. Kedua pembanding dilaporkan agar penurunan akibat kompresi sebelumnya tidak tersembunyi. Validation telah dipakai pada eksperimen terdahulu; test tidak dipakai.

Tidak ada pemasangan model atau perubahan routing otomatis. Setelah kualitas dinilai, tahap berikutnya ialah pipeline iPhone beberapa lagu/cold launch, memori dan pengemasan bobot sebelum integrasi aplikasi.
